# Comparison

## Loading Dataset

In [ ]:
from datasets import load_dataset
from config import DATASET_NAME

dataset = load_dataset(DATASET_NAME)
all_data = dataset["train"]

train_samples = [{"image": x["image_path"], "text": x["text"]} for x in all_data if x["split"] == "train"]
val_samples = [{"image": x["image_path"], "text": x["text"]} for x in all_data if x["split"] == "val"]
test_samples = [{"image": x["image_path"], "text": x["text"]} for x in all_data if x["split"] == "test"]

print("Successfully loaded dataset")
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

## Loading Pretrained and Fine-tuned models

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, GenerationConfig
from PIL import Image, ImageOps
from config import PT_MODEL_NAME, FT_MODEL_NAME, get_device
import torch

# Load pretrained model
pretrained_processor = TrOCRProcessor.from_pretrained(PT_MODEL_NAME)
pretrained_model = VisionEncoderDecoderModel.from_pretrained(PT_MODEL_NAME, torch_dtype=torch.float32)
pretrained_model.eval()

# Load fine-tuned model from Hub
finetuned_processor = TrOCRProcessor.from_pretrained(FT_MODEL_NAME)
finetuned_model = VisionEncoderDecoderModel.from_pretrained(FT_MODEL_NAME, torch_dtype=torch.float32)
finetuned_model.eval()

device = torch.device(get_device())

pretrained_model.to(device)
finetuned_model.to(device)

# Fix token IDs on pretrained model
pretrained_model.config.decoder_start_token_id = 1
pretrained_model.config.pad_token_id = 1
pretrained_model.config.eos_token_id = 2
pretrained_model.generation_config = GenerationConfig(
    decoder_start_token_id=1,
    eos_token_id=2,
    pad_token_id=1,
    max_new_tokens=64,
    forced_eos_token_id=None,
    forced_bos_token_id=None,
)

print("Successfully loaded models")
print(f"device:", device)

In [ ]:
def predict(model, processor, sample):
    image = sample["image"].convert("RGB")
    image = ImageOps.grayscale(image)
    image = ImageOps.autocontrast(image)
    image = image.convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device).float()
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


def compare(sample):
    pretrained_out = predict(pretrained_model, pretrained_processor, sample)
    finetuned_out = predict(finetuned_model,  finetuned_processor,  sample)
    print(f"Ground truth: {sample['text']}")
    print(f"Pretrained:   {pretrained_out}")
    print(f"Fine-tuned:   {finetuned_out}")
    print()

print("Loaded comparison functions")

## Training Process Viz

In [ ]:
import json
import matplotlib.pyplot as plt
import os

# Load all history files
def load_history(filename):
    with open(f"results/metrics/{filename}", "r") as f:
        return json.load(f)

history_1 = load_history("lr=5e-5_augTrue_5ep.json")
history_2 = load_history("lr=1e-5_augTrue_5ep.json")
history_3 = load_history("lr=1e-4_augTrue_5ep.json")
history_4 = load_history("lr=5e-5_augFalse_5ep.json")
history_best = load_history("lr=1e-5_augTrue_15ep.json")

os.makedirs("results/figures", exist_ok=True)

# Plot 1 — Loss curve for best experiment
plt.figure(figsize=(10, 5))
plt.plot(range(1, 16), history_best["train_loss"], marker="o", color="steelblue", label="Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Over 15 Epochs (lr=1e-5, augment=True)")
plt.xticks(range(1, 16))
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/loss_curve_best.png", dpi=150)
plt.show()

# Plot 2 — Val CER across all experiments
plt.figure(figsize=(10, 5))
plt.plot(range(1, 6), history_1["val_cer"], marker="o", label="lr=5e-5, augment=True")
plt.plot(range(1, 6), history_2["val_cer"], marker="o", label="lr=1e-5, augment=True")
plt.plot(range(1, 6), history_3["val_cer"], marker="o", label="lr=1e-4, augment=True")
plt.plot(range(1, 6), history_4["val_cer"], marker="o", label="lr=5e-5, augment=False")
plt.xlabel("Epoch")
plt.ylabel("Val CER")
plt.title("Validation CER Across All Experiments")
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/val_cer_all_experiments.png", dpi=150)
plt.show()